# Trabajo Práctico Final: Resolución de Rompecabezas con Encastres (Macho y Hembra)
## Cátedra de Procesamiento Digital de Imágenes (PDI)

¡Bienvenidos al notebook interactivo oficial del Trabajo Práctico Final!

En este tutorial aprenderán paso a paso las técnicas de PDI necesarias para resolver rompecabezas con encastres reales:
1. **Carga de piezas sobre fondo negro.**
2. **Binarización de la pieza (Fondo negro '0' vs. Pieza completa '255') mediante Otsu.**
3. **Detección de bordes y contornos externos con OpenCV.**
4. **Detección de esquinas y partición en los 4 lados (N, S, E, W).**
5. **Clasificación morfológica de encastres:**
   - `PLANO`: Borde exterior de la imagen original.
   - `MACHO`: Pestaña o saliente hacia afuera.
   - `HEMBRA`: Hueco o hendidura hacia adentro.
6. **Matcheo de forma de borde + continuidad de color y gradientes.**
7. **Reconstrucción de la grilla mediante Búsqueda Voraz con Backtracking.**

In [ ]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Añadir directorio raíz del TP para importar módulos de la cátedra
sys.path.append(os.path.abspath(".."))

from src.utils import load_image, assemble_puzzle
from src.shape_matcher import binarize_piece, extract_external_contour, analyze_piece_shape, compute_jigsaw_shape_compatibility
from src.baseline_solver import BaselineSolver
from src.evaluator import PuzzleEvaluator

%matplotlib inline
print("¡Entorno configurado correctamente!")

### Paso 1: Cargar una pieza con encastre (Fondo Negro)
Cada pieza se guarda con su forma irregular sobre un fondo negro puro `(0, 0, 0)`.

In [ ]:
puzzle_dir = "../dataset_ejemplos/puzzle_3x3_facil"
piece_path = os.path.join(puzzle_dir, "pieces", "piece_000.png")

pieza_0 = load_image(piece_path)

plt.figure(figsize=(6, 6))
plt.imshow(pieza_0)
plt.title("Pieza 0: Forma Irregular sobre Fondo Negro", fontsize=12, fontweight="bold")
plt.axis("off")
plt.show()

### Paso 2: Binarización de la Pieza (Otsu)

El primer objetivo de PDI es aislar la silueta completa de la pieza (blanco = 255) del fondo negro (0).
Utilizamos umbralización global con el algoritmo de **Otsu** y una pasada de cierre morfológico.

In [ ]:
# Binarización
mascara_binaria = binarize_piece(pieza_0, method="otsu")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(pieza_0)
axes[0].set_title("Imagen RGB Original", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(mascara_binaria, cmap="gray")
axes[1].set_title("Máscara Binarizada (Otsu)", fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

### Paso 3: Detección de Contornos y Clasificación (Macho / Hembra / Plano)

Aplicamos `cv2.findContours` para extraer el contorno exterior continuo de la pieza.
Luego, detectamos las 4 esquinas de la grilla y medimos la desviación respecto a la línea recta:

In [ ]:
# Análisis morfológico completo
analisis = analyze_piece_shape(pieza_0)
lados = analisis["sides"]

print("Clasificación Morfológica de los Lados de la Pieza 0:")
for lado in ["N", "E", "S", "W"]:
    tipo = lados[lado]["type"]
    print(f"  - Lado {lado}: {tipo}")

# Dibujar el contorno coloreando cada lado según su tipo
canvas = pieza_0.copy()
colores = {"MACHO": (0, 255, 0), "HEMBRA": (255, 0, 0), "PLANO": (255, 255, 0)}

for lado in ["N", "E", "S", "W"]:
    pts = lados[lado]["curve"].astype(np.int32)
    color = colores[lados[lado]["type"]]
    cv2.polylines(canvas, [pts], isClosed=False, color=color, thickness=4)

plt.figure(figsize=(7, 7))
plt.imshow(canvas)
plt.title("Contorno Segmentado\n(Verde = Macho, Rojo = Hembra, Amarillo = Plano)", fontsize=12, fontweight="bold")
plt.axis("off")
plt.show()

### Paso 4: Matcheo Geométrico de Encastres Complementarios

**Regla fundamental de ensamble:**
- Un lado `MACHO` **SOLO** puede encajar con un lado `HEMBRA`.
- Si comparamos `MACHO` con `MACHO` o `HEMBRA` con `HEMBRA`, el costo es infinito (incompatibles).
- Cuando son complementarios, calculamos la distancia entre sus perfiles 1D:

In [ ]:
# Carguemos la Pieza 3 (que en la solución está a la derecha de la Pieza 0)
pieza_3 = load_image(os.path.join(puzzle_dir, "pieces", "piece_003.png"))
analisis_3 = analyze_piece_shape(pieza_3)

borde_este_0 = analisis["sides"]["E"]
borde_oeste_3 = analisis_3["sides"]["W"]

print(f"Pieza 0 (Borde Este):  Tipo = {borde_este_0['type']}")
print(f"Pieza 3 (Borde Oeste): Tipo = {borde_oeste_3['type']}")

costo_forma = compute_jigsaw_shape_compatibility(borde_este_0, borde_oeste_3)
print(f"Costo de ajuste de encastre: {costo_forma:.2f} (Bajo = encaje perfecto)")

# Graficar perfiles complementarios superpuestos
plt.figure(figsize=(10, 4))
plt.plot(borde_este_0["profile"], label="Pieza 0 - Borde Este (Macho/Hembra)", color="green", linewidth=2.5)
plt.plot(-borde_oeste_3["profile"], label="Pieza 3 - Borde Oeste Invertido (Complementario)", color="red", linestyle="--", linewidth=2)
plt.title("Alineación y Complementariedad de Curvas de Encastre", fontsize=12, fontweight="bold")
plt.xlabel("Muestra normalizada (0-50)")
plt.ylabel("Desviación transversal respecto a la recta base (px)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Paso 5: Reconstrucción y Evaluación Oficial

Ejecutamos el solver combinando el matcheo de forma de los encastres con el algoritmo de búsqueda voraz y backtracking.

In [ ]:
solver = BaselineSolver(puzzle_dir)
grid_pred, rotations_pred = solver.solve()

# Guardar predicción y evaluar
pred_path = "test_jigsaw_pred.json"
solver.save_solution(grid_pred, rotations_pred, pred_path)

evaluator = PuzzleEvaluator(os.path.join(puzzle_dir, "ground_truth.json"))
resultados = evaluator.evaluate(pred_path, save_visual_path="reporte_jigsaw_eval.png")

# Mostrar reporte visual generado
reporte_img = load_image("reporte_jigsaw_eval.png")
plt.figure(figsize=(15, 5))
plt.imshow(reporte_img)
plt.axis("off")
plt.show()

### Paso 6: Análisis de Rayas Horizontales y Estimación de Orientación (FFT 2D / Gradientes)
Cuando el rompecabezas incluye un patrón de modulación periódica (rayas horizontales), cada pieza contiene información de su orientación absoluta.
Podemos estimar la inclinación de la pieza mediante el espectro de Fourier 2D o el histograma de orientaciones de gradientes Sobel.

In [ ]:
from src.stripe_analyzer import detect_stripe_orientation, detect_stripe_orientation_fft, rectify_piece_rotation, apply_horizontal_stripes
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Crear pieza sintética con rayas inclinadas 15 grados
sample_tile = piece_0.copy()
striped_tile = apply_horizontal_stripes(sample_tile, period=8, amplitude=0.35)
tilted_tile, _ = rectify_piece_rotation(striped_tile, 15.0)

# Estimar orientación con Sobel y con FFT
estimated_angle = detect_stripe_orientation(tilted_tile)
print(f"Ángulo de inclinación detectado: {estimated_angle:.1f}° (Esperado: 15.0°)")

# Rectificar pieza
rectified, _ = rectify_piece_rotation(tilted_tile, -estimated_angle)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(striped_tile)
axes[0].set_title("1. Rayas Horizontales (0°)")
axes[0].axis("off")

axes[1].imshow(tilted_tile)
axes[1].set_title(f"2. Pieza Inclinada (+15°)\nDetectado: {estimated_angle:.1f}°")
axes[1].axis("off")

axes[2].imshow(rectified)
axes[2].set_title("3. Pieza Rectificada (Deskewed)")
axes[2].axis("off")
plt.tight_layout()
plt.show()


### Paso 7: Comparación de Funciones de Curva Específicas (Standard, Circular, Random)
Los encastres analíticos pueden generarse con distintas funciones geométricas:
- **Standard**: bulbo suave con cuello cosenoidal.
- **Circular**: saliente redondeada / semicircular.
- **Random**: saliente asimétrica / ondulada.
Los alumnos deben verificar no solo que sea Macho con Hembra, sino que compartan la misma función de curva.

In [ ]:
from src.jigsaw_geometry import generate_tab_curve

p_start = (0, 0)
p_end = (100, 0)

pts_std = generate_tab_curve(p_start, p_end, tab_type=1, profile_type="standard")
pts_circ = generate_tab_curve(p_start, p_end, tab_type=1, profile_type="circular")
pts_rnd = generate_tab_curve(p_start, p_end, tab_type=1, profile_type="random")

plt.figure(figsize=(10, 4))
plt.plot(pts_std[:, 0], -pts_std[:, 1], label="Standard (Bulbo Cosenoidal)", color="blue", lw=2)
plt.plot(pts_circ[:, 0], -pts_circ[:, 1], label="Circular (Semicírculo)", color="red", lw=2, linestyle="--")
plt.plot(pts_rnd[:, 0], -pts_rnd[:, 1], label="Random (Asimétrica/Ondulada)", color="green", lw=2, linestyle="-.")
plt.title("Comparación de Firmas de Curva de Encastre Específicas")
plt.xlabel("Longitud de Borde (px)")
plt.ylabel("Desviación Normal (px)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
